# Email Spam Detection

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df=pd.read_csv('emails.csv')
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [8]:
df.isnull().sum()

text    0
spam    0
dtype: int64

In [11]:
df['spam'].value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

In [12]:
df['spam'].unique()

array([1, 0])

In [13]:
# Convert all text data to lowercase
df['text']=df['text'].str.lower()

In [14]:
df['text']

0       subject: naturally irresistible your corporate...
1       subject: the stock trading gunslinger  fanny i...
2       subject: unbelievable new homes made easy  im ...
3       subject: 4 color printing special  request add...
4       subject: do not have money , get software cds ...
                              ...                        
5723    subject: re : research and development charges...
5724    subject: re : receipts from visit  jim ,  than...
5725    subject: re : enron case study update  wow ! a...
5726    subject: re : interest  david ,  please , call...
5727    subject: news : aurora 5 . 2 update  aurora ve...
Name: text, Length: 5728, dtype: object

In [15]:
# Remove punctuation from text data
import string

df['text'] = df['text'].str.replace(
    f"[{string.punctuation}]", "", regex=True
)

In [17]:
df['text'].head()

0    subject naturally irresistible your corporate ...
1    subject the stock trading gunslinger  fanny is...
2    subject unbelievable new homes made easy  im w...
3    subject 4 color printing special  request addi...
4    subject do not have money  get software cds fr...
Name: text, dtype: object

In [19]:
df['text']=df['text'].str.replace('subject ', '', regex=False)
df['text']

0       naturally irresistible your corporate identity...
1       the stock trading gunslinger  fanny is merrill...
2       unbelievable new homes made easy  im wanting t...
3       4 color printing special  request additional i...
4       do not have money  get software cds from here ...
                              ...                        
5723    re  research and development charges to gpg  h...
5724    re  receipts from visit  jim   thanks again fo...
5725    re  enron case study update  wow  all on the s...
5726    re  interest  david   please  call shirley cre...
5727    news  aurora 5  2 update  aurora version 5  2 ...
Name: text, Length: 5728, dtype: object

In [20]:
# Remove numbers from text data
df['text'] = df['text'].str.replace(r'\d+', '', regex=True)

In [21]:
# Remove URLs and links from text data
df['text'] = df['text'].str.replace(r'https?://\S+|www\.\S+', '', regex=True)

In [23]:
df.head()

,text,spam
0,naturally irresistible your corporate identity...,1
1,the stock trading gunslinger fanny is merrill...,1
2,unbelievable new homes made easy im wanting t...,1
3,color printing special request additional in...,1
4,do not have money get software cds from here ...,1


In [24]:
# Remove emojis and special characters from text data
df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)

In [25]:
# Import NLTK and the English stop words dataset
import nltk
from nltk.corpus import stopwords

# Download the stop words list (only needed once)
nltk.download('stopwords')

# Store English stop words in a set for faster lookup
stop_words = set(stopwords.words('english'))

# Split each text into words, remove stop words, and join the remaining words
df['text'] = df['text'].apply(
    lambda text: ' '.join(
        word for word in text.split()  # Split text into individual words
        if word not in stop_words      # Keep only words that are not stop words
    )
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [26]:
df.head()

,text,spam
0,naturally irresistible corporate identity lt r...,1
1,stock trading gunslinger fanny merrill muzo co...,1
2,unbelievable new homes made easy im wanting sh...,1
3,color printing special request additional info...,1
4,money get software cds software compatibility ...,1


In [28]:
df.shape

(5728, 2)

In [30]:
# Import train_test_split to divide the dataset
from sklearn.model_selection import train_test_split

# X = input text
# y = target labels (spam)
# test_size=0.20 means 20% data for testing and 80% for training
# random_state=42 ensures the same split every time
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['spam'],
    test_size=0.20,
    random_state=42
)

# Check the number of samples in training and testing data
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4582
Testing samples: 1146


In [31]:
# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Create the CountVectorizer object
cv = CountVectorizer()

# Learn the vocabulary from training text and convert it into word counts
X_train_cv = cv.fit_transform(X_train)

# Convert test text using the same vocabulary learned from training data
X_test_cv = cv.transform(X_test)

# Display the dimensions of the converted data
print("Training data shape:", X_train_cv.shape)
print("Testing data shape:", X_test_cv.shape)

Training data shape: (4582, 30350)
Testing data shape: (1146, 30350)


In [32]:
# Import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer object
tfidf = TfidfVectorizer()

# Learn vocabulary and calculate TF-IDF features from training text
X_train_tfidf = tfidf.fit_transform(X_train)

# Apply the learned vocabulary and weights to testing text
X_test_tfidf = tfidf.transform(X_test)

# Display the dimensions of the converted data
print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (4582, 30350)
Testing data shape: (1146, 30350)


In [33]:
# Import the three classification algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

# Create a dictionary containing the models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC()
}

# Store the accuracy of each model for comparison
results = {}

# Train and evaluate each model
for name, model in models.items():

    # Train the model using TF-IDF features and emotion labels
    model.fit(X_train_tfidf, y_train)

    # Predict emotions for the test data
    y_pred = model.predict(X_test_tfidf)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Save accuracy for the final comparison
    results[name] = accuracy

    # Print the model name and its accuracy
    print("=" * 60)
    print("Model:", name)
    print("Accuracy:", round(accuracy, 4))

    # Display precision, recall, F1-score, and support for each emotion
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

Model: Naive Bayes
Accuracy: 0.8866

Classification Report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       856
           1       1.00      0.55      0.71       290

    accuracy                           0.89      1146
   macro avg       0.93      0.78      0.82      1146
weighted avg       0.90      0.89      0.87      1146

Model: Logistic Regression
Accuracy: 0.9773

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       856
           1       0.99      0.92      0.95       290

    accuracy                           0.98      1146
   macro avg       0.98      0.96      0.97      1146
weighted avg       0.98      0.98      0.98      1146

Model: SVM
Accuracy: 0.9939

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       856
           1       0.99      0.98      0.99       290

  

In [34]:
# Import Linear Support Vector Machine
from sklearn.svm import LinearSVC

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

# Create the SVM model
svm_model = LinearSVC()

# Train SVM using the TF-IDF training data
svm_model.fit(X_train_tfidf, y_train)

# Predict emotions for the test data
svm_pred = svm_model.predict(X_test_tfidf)

# Calculate and display accuracy
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

# Display precision, recall, and F1-score for each emotion
print(classification_report(y_test, svm_pred, zero_division=0))

SVM Accuracy: 0.993891797556719
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       856
           1       0.99      0.98      0.99       290

    accuracy                           0.99      1146
   macro avg       0.99      0.99      0.99      1146
weighted avg       0.99      0.99      0.99      1146

